In [ ]:
import sys
import os
from pathlib import Path  # noqa: F401

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

import numpy as np  # noqa: E402
import seaborn as sns  # noqa: E402

from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    ExclusionCategories,
    ExperimentNames,
    MusicTypeVariants,
)
from src.analysis.mean_variance import (  # noqa: E402
    FREQUENCY_BANDS,
    compute_band_intersubject_stats,
    compute_pairwise_isc_matrices,
)
from src.visualization.mean_variance_plots import (  # noqa: E402
    plot_band_timeseries,
    plot_band_variance_distributions,
    plot_isc_matrices,
    plot_band_windowed_analysis,
)
from scripts.analysis_common import load_analyzers, analyzers_to_datasets  # noqa: E402
from src.definitions.constants import ProjectPaths  # noqa: E402

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
print("Setup complete.")

# EEG Intersubject Mean-Variance Synchrony Analysis — Per-Frequency-Band

This notebook implements the **per-frequency-band** intersubject mean-variance synchrony pipeline:

| Band  | Range |
|-------|-------|
| Delta | 1–4 Hz |
| Theta | 4–8 Hz |
| Alpha | 8–13 Hz |
| Beta  | 13–30 Hz |
| Gamma | 30–70 Hz |

Covers band-specific time series, variance distributions, pairwise ISC matrices, and windowed synchrony per band.

All computation uses `src.analysis.mean_variance` and all visualisation uses
`src.visualization.mean_variance_plots`.

> **Parameters to tweak:** `CONDITION`, `MUSIC_TYPES`, `WINDOW_SEC`, `SYNC_PERCENTILE` in the
> *Configuration* cell below.

## Configuration

In [ ]:
# ── Dataset switch ─────────────────────────────────────────────────────────
# Choose which experiment to analyse:
#   ExperimentNames.PSILO_MUSIC → psilocybin music-listening (CLASSIC / PSYTRANCE)
#   ExperimentNames.ASSR        → auditory steady-state response (no music dimension)
EXPERIMENT_NAME = ExperimentNames.PSILO_MUSIC  # or ExperimentNames.ASSR

if EXPERIMENT_NAME == ExperimentNames.ASSR:
    # ASSR has no music dimension; uses a single placeholder "music type".
    MUSIC_TYPES = [MusicTypeVariants.ASSR]
else:
    MUSIC_TYPES = [MusicTypeVariants.CLASSICAL, MusicTypeVariants.PSYTRANCE]

# ── Experiment configuration ─────────────────────────────────────────────────
CONDITION = ConditionVariants.PLACEBO
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]

# ── Windowed synchrony parameters ────────────────────────────────────────────
WINDOW_SEC = 2.0  # window length in seconds
STEP_SEC = WINDOW_SEC / 2  # 50% overlap between successive windows
SYNC_PERCENTILE = 10.0  # windows with variance < this percentile are "sync candidates"

# ── Data processing flag ──────────────────────────────────────────────────────
# Set True to load raw EDF files, resample, stack, and save before analysis.
# Keep False to use already-saved concatenated arrays.
process_and_save_data = False

# ── Plot saving ──────────────────────────────────────────────────────────────
# Plots are saved into a per-experiment 'plots/' subdirectory next to this
# notebook, so the datasets don't overwrite each other.
# Set SAVE_PLOTS=False to only display figures inline without saving.
SAVE_PLOTS = True
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "01-raw-mean-variance-analysis"
    / "plots"
    / EXPERIMENT_NAME.value
    / "bands"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Experiment: {EXPERIMENT_NAME.value}")
print(f"Plots will be saved to: {PLOTS_DIR}")

## Data Loading

In [ ]:
# Load (or process-and-save) one EEGSummarizedAnalyzer per music type,
# normalise the data to z-scores (axis=2), and convert to AnalysisData.
analyzers = load_analyzers(
    MUSIC_TYPES,
    CONDITION,
    EXCLUSION_CATEGORIES,
    process_and_save_data,
    normalize_data=True,
    experiment_name=EXPERIMENT_NAME,
)
datasets = analyzers_to_datasets(analyzers)
print("Available datasets:", list(datasets.keys()))

## Dataset Selection

Change `LABEL` to switch between music types.  The remaining cells use `ad` and the
derived variables `n_subjects`, `n_channels`, `n_times`.

In [ ]:
# Default to the first configured music type (ASSR has only one).
# For the MUSIC dataset, switch by uncommenting the desired line.
LABEL = MUSIC_TYPES[0].value
# LABEL = MusicTypeVariants.CLASSICAL.value
# LABEL = MusicTypeVariants.PSYTRANCE.value

ad = datasets[f"{CONDITION.value}_{LABEL}"]
n_subjects, n_channels, n_times = ad.data.shape
print(f"Dataset : {LABEL}")
print(f"Shape   : {ad.data.shape}  (subjects × channels × time points)")
print(f"Duration: {n_times / ad.sfreq:.1f} s  @  {ad.sfreq} Hz")

---
## Part 2 — Per-Frequency-Band Analysis

The same analyses are repeated independently for each frequency band after
bandpass-filtering the z-scored data:

| Band  | Range |
|-------|-------|
| Delta | 1–4 Hz |
| Theta | 4–8 Hz |
| Alpha | 8–13 Hz |
| Beta  | 13–30 Hz |
| Gamma | 30–70 Hz |

### 2.1  Compute per-band intersubject statistics

`compute_band_intersubject_stats` applies `compute_intersubject_stats` to the
bandpass-filtered copy of `ad` for every entry in `FREQUENCY_BANDS`.

In [ ]:
band_stats = compute_band_intersubject_stats(ad)

print(f"{'Band':8s}  {'mean var_t':>12s}  {'mean mean_t':>12s}")
print("-" * 36)
for band, st in band_stats.items():
    print(f"{band:8s}  {st['var_t'].mean():12.4f}  {st['mean_t'].mean():12.4f}")

### 2.2  Per-band time-series overview

Five-row two-column figure.
* **Left column** — group-mean signal with per-subject traces and ±1 SD per band
* **Right column** — channel-averaged intersubject variance with synchrony threshold

In [ ]:
fig_band_ts = plot_band_timeseries(
    band_stats,
    ad.sfreq,
    label=LABEL,
    sync_percentile=SYNC_PERCENTILE,
    save_path=PLOTS_DIR / "band_timeseries.png" if SAVE_PLOTS else None,
)

### 2.3  Per-band intersubject variance distributions

One histogram per band in a 2-column grid, each clipped at the 99th percentile.

In [ ]:
fig_band_dist = plot_band_variance_distributions(
    band_stats,
    label=LABEL,
    save_path=PLOTS_DIR / "band_variance_distributions.png" if SAVE_PLOTS else None,
)

### 2.4  Pairwise inter-subject correlation (ISC) matrices

For z-scored data the mean product across channels and time equals the mean Pearson
correlation.  `compute_pairwise_isc_matrices` computes this for every subject pair
and frequency band.

In [ ]:
# Build {band: filtered_data_array} for the ISC computation
band_data_arrays = {
    band: ad.filter_to_band(l_freq, h_freq).data
    for band, (l_freq, h_freq) in FREQUENCY_BANDS.items()
}
isc_matrices = compute_pairwise_isc_matrices(band_data_arrays)

mask = ~np.eye(n_subjects, dtype=bool)
print(f"{'Band':8s}  {'off-diag mean ISC':>18s}")
print("-" * 30)
for band, mat in isc_matrices.items():
    print(f"{band:8s}  {mat[mask].mean():18.4f}")

In [ ]:
fig_isc = plot_isc_matrices(
    isc_matrices,
    n_subjects=n_subjects,
    label=LABEL,
    save_path=PLOTS_DIR / "isc_matrices.png" if SAVE_PLOTS else None,
)

### 2.5  Per-band windowed synchrony analysis

Two outputs:
1. **Summary figure** — all bands in one multi-row plot with windowed variance and
   synchrony candidate spans highlighted in green
2. **Per-band detailed figures** — one per band with continuous variance, windowed
   step function, sync threshold, and per-electrode variance heatmap

In [ ]:
fig_band_win_bar, fig_band_win_summary, per_band_figs = plot_band_windowed_analysis(
    band_stats,
    ad.sfreq,
    label=LABEL,
    window_sec=WINDOW_SEC,
    sync_percentile=SYNC_PERCENTILE,
    step_sec=STEP_SEC,
    save_path_bar=PLOTS_DIR / "band_windowed_bar.png" if SAVE_PLOTS else None,
    save_path_summary=PLOTS_DIR / "band_windowed_summary.png" if SAVE_PLOTS else None,
    save_path_per_band_dir=PLOTS_DIR / "windowed_per_band" if SAVE_PLOTS else None,
)